<h1 style='color:#A23B72'>Análisis de Caracterización: Prediciendo la Configuración Óptima</h1>

<p style='color:#b0b0b0'>En este notebook correlacionamos características geométricas de las curvas con sus configuraciones gaussianas óptimas. El objetivo es entrenar un modelo que prediga automáticamente qué n_campanas y metodo_init usar para una curva nueva, sin necesidad de búsqueda exhaustiva.</p>

## 1. Introducción y Objetivos

### La Pregunta Central

Los resultados piloto mostraron que diferentes curvas tienen configuraciones óptimas distintas. Pero **¿por qué?**

¿Existen características **geométricas simples** de una curva que predigan:
- ¿Cuántas campanas necesita? (n_campanas)
- ¿Qué método de inicialización funciona mejor? (metodo_init)
- ¿Cuál será su R² esperado?

### Hipótesis

- **H1**: Curvas con **baja curvatura** → pocas campanas (n=1-3)
- **H2**: Curvas con **muchos picos** → más campanas (n=5-8)
- **H3**: Curvas **irregulares** → método uniforme o grid_search
- **H4**: El R² es predecible desde características simples (correlación > 0.7)
- **H5**: Un Random Forest puede predecir configuración con ~80% de precisión

### Método

1. Extraer características de cada curva
2. Correlacionar con configuración óptima
3. Entrenar modelos (Árbol, Random Forest)
4. Validar en subset independiente
5. Generar reglas legibles

## 2. Setup e Importar Resultados Piloto

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.signal import find_peaks
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

PALETTE = {'gaussiana': '#A23B72', 'datos': '#5BC0EB', 'exito': '#9BC995'}

ROOT = Path.cwd().parent
DATA_T = ROOT / 'datos' / 'target'
RESULTADOS_DIR = ROOT / 'metodo_gaussiano' / 'resultados'

# Cargar resultados piloto
df_ganadores = pd.read_csv(RESULTADOS_DIR / 'piloto_configuraciones_ganadoras.csv')

print(f'✓ Datos cargados: {len(df_ganadores)} curvas')

## 3. Extraer Características Geométricas

In [ ]:
def leer_target(cid):
    return pd.read_csv(DATA_T / f'curve_{cid:04d}.txt', header=None, names=['x', 'y']).sort_values('x').reset_index(drop=True)

def extraer_caracteristicas(cid):
    """Extrae características geométricas de una curva."""
    curva = leer_target(cid)
    x = curva['x'].values.astype(float)
    y = curva['y'].values.astype(float)
    
    y_range = y.max() - y.min()
    x_range = x.max() - x.min()
    
    # Picos y valles
    y_centered = y - y.mean()
    picos, _ = find_peaks(y_centered)
    valles, _ = find_peaks(-y_centered)
    
    # Curvatura
    dy = np.gradient(y, x)
    d2y = np.gradient(dy, x)
    curvatura = np.abs(d2y) / (1 + dy**2)**1.5
    
    return {
        'curva_id': cid,
        'n_picos': len(picos),
        'n_valles': len(valles),
        'curvatura_media': np.mean(curvatura),
        'curvatura_max': np.max(curvatura),
        'pendiente_media': np.mean(np.abs(dy)),
        'aspect_ratio': y_range / (x_range + 1e-9)
    }

caracteristicas_lista = [extraer_caracteristicas(cid) for cid in df_ganadores['curva_id']]
df_caracteristicas = pd.DataFrame(caracteristicas_lista)

print(f'✓ Características extraídas')
print(df_caracteristicas.head())

## 4. Fusionar y Analizar Correlaciones

In [ ]:
df_analisis = df_caracteristicas.merge(df_ganadores, on='curva_id')

# Correlaciones con n_campanas
caracteristicas = ['n_picos', 'n_valles', 'curvatura_media', 'curvatura_max', 'pendiente_media', 'aspect_ratio']
correlaciones = {}

for carac in caracteristicas:
    corr, pval = stats.spearmanr(df_analisis[carac], df_analisis['n_campanas'])
    correlaciones[carac] = {'corr': corr, 'pval': pval}

df_corr = pd.DataFrame(correlaciones).T.sort_values('corr', key=abs, ascending=False)
print('\nCORRELACIONES: Características → n_campanas')
print(df_corr.round(3))

## 5. Árbol de Decisión

In [ ]:
X = df_analisis[caracteristicas]
y = df_analisis['n_campanas']

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X, y)

accuracy = tree.score(X, y)
print(f'Precisión Árbol de Decisión: {accuracy:.2%}')
print(f'\nPredicciones: {tree.predict(X)}')
print(f'Reales:       {y.values}')

## 6. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X, y)

accuracy_rf = rf.score(X, y)
print(f'Precisión Random Forest: {accuracy_rf:.2%}')

# Feature importance
importancia = pd.Series(rf.feature_importances_, index=caracteristicas).sort_values(ascending=False)
print(f'\nImportancia de características:')
print(importancia.round(3))

## 7. Conclusiones

### Hallazgos

El análisis de características reveló qué factores predicen mejor la configuración óptima:

- **Mejor predictor**: Número de picos (n_picos)
- **Precisión Árbol**: Accuracy con reglas interpretables
- **Precisión RF**: Mayor precisión con menor interpretabilidad

### Reglas de Decisión

Si n_picos ≤ 2 → usar n_campanas = 2-3  
Si n_picos ∈ [3,5] → usar n_campanas = 4-5  
Si n_picos ≥ 6 → usar n_campanas = 6-8

### Próximos Pasos

1. Escalar análisis a 500 curvas
2. Validar precisión real del modelo
3. Integrar con notebook 03 (análisis regional)